# Stage 2 — selective parser + contract checks (contract-check-v1) + combined 3B repair
Same pipeline, models, adapters, prompts and limits as the completed combined 3B selective run (74 correct / 1 incorrect accepted / 25 withheld). One change: after the unchanged first parse, input-only checks inspect the contract. When a check fires, the same parser gets **one** correction turn with the issues. A contract that still has a proven inconsistency is withheld with its reason; evidence-only issues never cause withholding. Checks use only the request, input data and the predicted contract; reference programs and gold contracts remain evaluator-only.

Inputs: **stage2_contract_check_source.zip** and the same selected adapter dataset as the completed run (selective_facts_1_5b.zip). Enable Internet, select T4 x2, then Save Version → Run All. The notebook runs a two-task smoke, 20 development tasks, then 100 diagnostic test tasks, each fresh. Download **stage2_contract_check_selective_checked.zip** and the executed notebook.

The checks were designed after inspecting failures on these test tasks, so test gains are diagnostic, not independent evidence. Records keep the first and final contracts, every issue, the correction prompt/reply, its tokens and the decision.


In [1]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('run_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=None
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for p in RUN_DIR.iterdir():
            if p.is_file() and p.suffix in ['.zip','.log','.json','.txt']:shutil.copy2(p,DRIVE_RESULTS/p.name)
        print('Drive results:',DRIVE_RESULTS)
print('Run folder:',RUN_DIR)

METHOD='selective'
for name in ['stage2_contract_check_source.zip','selective_facts_1_5b.zip']:
    matches=list(INPUT.rglob(name));print(('found ZIP: ' if matches else 'No intact ZIP (may be extracted): ')+name,*matches)
print('The next cells verify source files and adapter contents, including automatically extracted datasets.')


Run folder: /kaggle/working/run_20261115_102436_183903
No intact ZIP (may be extracted): stage2_contract_check_source.zip
No intact ZIP (may be extracted): selective_facts_1_5b.zip
The next cells verify source files and adapter contents, including automatically extracted datasets.


In [2]:
MANIFEST_SHA='971a9a25bf8a0860005b409705976159ac58505839ba43c74f043966099ee324'
ROOT=WORK/('stage2_contract_check_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source ZIP version'
    manifest=json.loads(raw);assert manifest['experiment']=='stage2-contract-check-3b-fp16-v1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source/input: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==data,f'Changed working source: {name}'
        else:path.write_bytes(data)
    sidecar=ROOT/'_stage2_contract_check_manifest.json'
    if sidecar.exists():assert sidecar.read_bytes()==raw,'Changed source manifest'
    else:sidecar.write_bytes(raw)
    print('Verified fresh-run source/data:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_contract_check_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='stage2_contract_check_source/';name=prefix+'manifest.json'
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place stage2_contract_check_source.zip in the input folder.'
PACKAGES=WORK/'stage2_contract_check_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


Verified fresh-run source/data: 279 files 971a9a25bf8a0860005b409705976159ac58505839ba43c74f043966099ee324


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.33.3 requires PyYAML>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
d

In [3]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(); assert all(torch.cuda.get_device_properties(i).total_memory/2**30>=12 for i in range(torch.cuda.device_count())); print(torch.__version__,[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','eval.stage2_combined_3b.preflight'],cwd=ROOT,env=ENV,check=True)
path_file=RUN_DIR/'adapter_path.txt'
subprocess.run([sys.executable,'-m','eval.stage2_combined_3b.adapters','--method',METHOD,'--input',str(INPUT),'--work',str(WORK),'--out',str(path_file)],cwd=ROOT,env=ENV,check=True)
ADAPTER=Path(path_file.read_text());save_results()


Removing optional torchao 0.10.0: incompatible with PEFT 0.21.2; this workflow does not use torchao.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
{"torchao": null, "action": "removed_incompatible", "previous_version": "0.10.0"}
{"probe": "cpu_lora_injection_and_forward", "trainable_parameters": 16, "status": "passed"}
2.11.0+cu130 ['Tesla T4', 'Tesla T4']
{"event": "attention_visible_devices", "devices": [0, 1], "cuda_visible_devices": null}
{"event": "attention_shape", "device": "cuda:0", "query_shape": [1, 4, 32, 128], "original_key_shape": [1, 2, 32, 128], "expanded_key_shape": [1, 4, 32, 128], "dtype": "torch.float16", "native_gqa_expanded": true, "stage": "prefill", "has_mask": false, "is_causal": true, "backend": "EFFICIENT_ATTENTION", "math_fallback_allowed": false}
{"event": "kernel_self_test_passed", "device": "cuda:0", "case": "causal_prefill", "max_absolute_error": 0.001953125, "rtol": 0.005, "atol": 0.005

In [4]:
DEV_COUNT=20
def execute(smoke,split='test'):
    name='stage2_contract_check_'+METHOD+'_checked'+('_smoke' if smoke else '_dev' if split=='dev' else '')
    out=RUN_DIR/name;assert not out.exists(),'Use a fresh run folder.'
    command=[sys.executable,'-m','eval.stage2_contract_check.runner','--method',METHOD,'--preset','proposed','--arm','checked','--adapter',str(ADAPTER),'--out',str(out),'--packages',str(PACKAGES),'--split',split]
    if smoke:command+=['--smoke']
    log=RUN_DIR/(name+'.log')
    try:
        with log.open('w') as f:
            process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end='');f.write(line);f.flush()
            code=process.wait()
    finally:save_results()
    assert code==0,f'Run failed: inspect {log} and the partial ZIP. A partial job is not a result.'
    assert json.loads((out/'completion.json').read_text())['complete']
    assert len(list((out/'tasks').glob('*.json')))==(2 if smoke else DEV_COUNT if split=='dev' else 100)
    return out.with_suffix('.zip')
SMOKE=execute(True)
DEV=execute(False,split='dev')
print('Development summary:',(DEV.with_suffix('')/'summary.json').read_text())
print('Smoke and end-to-end development completed. Inspect the development summary; the next cell runs 100 diagnostic test tasks freshly.')



Loading weights: 100%|██████████| 338/338 [00:01<00:00, 170.26it/s]
PARSE 1/2 adult_col_001: ok
PARSE 2/2 titanic_miss_004: ok

Fetching 2 files: 100%|██████████| 2/2 [00:32<00:00, 16.40s/it]

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 124.59it/s]
GENERATE 1/2 adult_col_001
GENERATE 2/2 titanic_miss_004

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 126.54it/s]
DONE 1/2 adult_col_001: ACCEPT
DONE 2/2 titanic_miss_004: ACCEPT

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 168.50it/s]
PARSE 1/20 adult_col_006: contract_check_failed
PARSE 2/20 adult_conv_006: ok
PARSE 3/20 adult_dedup_006: ok
PARSE 4/20 adult_filt_006: ok
PARSE 5/20 adult_miss_006: ok
PARSE 6/20 airbnb_col_006: ok
PARSE 7/20 airbnb_conv_006: ok
PARSE 8/20 airbnb_dedup_006: ok
PARSE 9/20 airbnb_filt_006: contract_check_failed
PARSE 10/20 airbnb_miss_006: contract_check_failed
PARSE 11/20 taxi_col_006: ok
PARSE 12/20 taxi_conv_006: ok
PARSE 13/20 taxi_dedup_006: ok
PARSE 14/20 taxi_filt_006

In [5]:
BUNDLE=execute(False)
from IPython.display import display,FileLink
display(FileLink(str(BUNDLE)))
if DRIVE_RESULTS is not None:print('Drive copy:',DRIVE_RESULTS/BUNDLE.name)
print('COMPLETE: download the full ZIP and this executed notebook. Send it back for paired analysis against the completed combined 3B control (tools/analyze_contract_check_run.py).')



Loading weights: 100%|██████████| 338/338 [00:02<00:00, 168.37it/s]
PARSE 1/100 adult_col_001: ok
PARSE 2/100 adult_col_002: ok
PARSE 3/100 adult_col_003: ok
PARSE 4/100 adult_col_004: ok
PARSE 5/100 adult_col_005: ok
PARSE 6/100 adult_conv_001: ok
PARSE 7/100 adult_conv_002: ok
PARSE 8/100 adult_conv_003: contract_check_failed
PARSE 9/100 adult_conv_004: ok
PARSE 10/100 adult_conv_005: ok
PARSE 11/100 adult_dedup_001: ok
PARSE 12/100 adult_dedup_002: ok
PARSE 13/100 adult_dedup_003: ok
PARSE 14/100 adult_dedup_004: ok
PARSE 15/100 adult_dedup_005: ok
PARSE 16/100 adult_filt_001: ok
PARSE 17/100 adult_filt_002: ok
PARSE 18/100 adult_filt_003: ok
PARSE 19/100 adult_filt_004: contract_check_failed
PARSE 20/100 adult_filt_005: contract_check_failed
PARSE 21/100 adult_miss_001: ok
PARSE 22/100 adult_miss_002: ok
PARSE 23/100 adult_miss_003: ok
PARSE 24/100 adult_miss_004: ok
PARSE 25/100 adult_miss_005: ok
PARSE 26/100 airbnb_col_001: ok
PARSE 27/100 airbnb_col_002: ok
PARSE 28/100 airbnb

/kaggle/working/run_20261115_102436_183903/stage2_contract_check_selective_checked.zip

COMPLETE: download the full ZIP and this executed notebook. Send it back for paired analysis against the completed combined 3B control (tools/analyze_contract_check_run.py).
